In [ ]:
import os
import warnings
import boto3
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score
from scipy.stats import ks_2samp

#### Functions

In [ ]:
# compute metrics by group
def compute_metrics_by_group(df_data, str_group_col, str_pred_col, str_target):
    list_rows = []
    for str_group, df_group in df_data.groupby(str_group_col):
        arr_y_true = df_group[str_target].values
        arr_y_pred = df_group[str_pred_col].values
        dict_row = {
            'str_group': str_group,
            'int_n': len(df_group),
            'flt_target_mean': arr_y_true.mean(),
            'flt_pred_mean': arr_y_pred.mean(),
            'flt_pred_median': np.median(arr_y_pred),
            'flt_auc': roc_auc_score(arr_y_true, arr_y_pred),
        }
        list_rows.append(dict_row)
    return pd.DataFrame(list_rows)

In [ ]:
# plot kde of predictions by group
def plot_kde_by_group(df_data, str_group_col, str_pred_col, str_filename='output/kde_by_group.png'):
    warnings.filterwarnings('ignore')
    fig, ax = plt.subplots(figsize=(10, 5))
    list_colors = ['steelblue', 'salmon']
    for (str_group, df_group), str_color in zip(df_data.groupby(str_group_col), list_colors):
        flt_median = np.median(df_group[str_pred_col].values)
        sns.kdeplot(df_group[str_pred_col], ax=ax, color=str_color, linewidth=2, label=f'{str_group} (Median={flt_median:.4f})', fill=True, alpha=0.2)
    ax.set_title(f'KDE of Predictions by {str_group_col}', fontsize=16)
    ax.set_xlabel('Predicted Probability', fontsize=12)
    ax.set_ylabel('Density', fontsize=12)
    ax.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(str_filename, dpi=300)
    plt.show()
    warnings.filterwarnings('default')

In [ ]:
# plot metric comparison by group
def plot_metric_comparison(df_metrics, str_metric, str_title, str_filename):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(df_metrics['str_group'].astype(str), df_metrics[str_metric], color='steelblue', edgecolor='black')
    ax.set_title(str_title, fontsize=16)
    ax.set_xlabel('Age Group', fontsize=12)
    ax.set_ylabel(str_metric, fontsize=12)
    # annotate bars
    for i, flt_val in enumerate(df_metrics[str_metric]):
        ax.text(i, flt_val + (ax.get_ylim()[1] * 0.01), f'{flt_val:.4f}', ha='center', fontsize=11)
    ax.set_ylim(top=ax.get_ylim()[1] * 1.15)
    plt.tight_layout()
    plt.savefig(str_filename, dpi=300)
    plt.show()

#### Constants

In [ ]:
# bucket
str_bucket = os.getcwd().split('/')[4].replace('_', '-')
print(f'Bucket: {str_bucket}')

# step
str_step = os.getcwd().split('/')[-1]
print(f'Step: {str_step}')

# s3 input path
str_s3_input = f's3://{str_bucket}/03_preprocessing'
print(f'S3 Input: {str_s3_input}')

# target
str_target = 'default_12m'
print(f'Target: {str_target}')

# model and feature columns
str_model_path = '../04_model/output/xgboost_model.joblib'
str_feature_cols_path = '../04_model/output/feature_cols.joblib'

# protected class column and threshold
str_age_col = 'int_age'
int_age_threshold = 60
print(f'Age threshold: {int_age_threshold}')

# output directory
os.makedirs('output', exist_ok=True)

#### Read Data and Model

In [ ]:
# read test data only
df_test = pd.read_parquet(f'{str_s3_input}/df_test_clean.parquet')

# load model and feature columns
model = joblib.load(str_model_path)
list_feature_cols = joblib.load(str_feature_cols_path)

print(f'Test: {df_test.shape}')
print(f'Features: {len(list_feature_cols)}')

#### Define Age Groups

Under ECOA (Equal Credit Opportunity Act), age is a protected class. Although `int_age` was excluded from model features for compliance, we must verify the model does not produce disparate outcomes across age groups. Applicants are segmented into under 60 and 60+ for this analysis.

In [ ]:
# generate predictions
df_test['flt_pred'] = model.predict_proba(df_test[list_feature_cols].values)[:, 1]

# create age group
df_test['str_age_group'] = np.where(df_test[str_age_col] >= int_age_threshold, f'>={int_age_threshold}', f'<{int_age_threshold}')

# print group counts
print(df_test['str_age_group'].value_counts())

#### Metrics by Age Group

Compare sample size, actual default rate, mean/median predicted probability, and AUC between age groups. Large differences in predicted probabilities or AUC may indicate disparate impact.

In [ ]:
df_di_metrics = compute_metrics_by_group(df_test, 'str_age_group', 'flt_pred', str_target)
df_di_metrics.to_csv('output/disparate_impact_metrics.csv', index=False)
df_di_metrics

#### Mean Predicted Probability by Age Group

In [ ]:
plot_metric_comparison(df_di_metrics, 'flt_pred_mean', 'Mean Predicted Probability by Age Group', 'output/pred_mean_by_age.png')

#### Actual Default Rate by Age Group

In [ ]:
plot_metric_comparison(df_di_metrics, 'flt_target_mean', 'Actual Default Rate by Age Group', 'output/target_mean_by_age.png')

#### AUC by Age Group

In [ ]:
plot_metric_comparison(df_di_metrics, 'flt_auc', 'AUC by Age Group', 'output/auc_by_age.png')

#### KDE of Predictions by Age Group

Comparing the distribution of predicted probabilities between age groups. Similar distributions suggest the model treats both groups comparably. A significant shift would indicate the model may be producing disparate outcomes.

In [ ]:
plot_kde_by_group(df_test, 'str_age_group', 'flt_pred')

#### KS Test

The Kolmogorov-Smirnov test formally tests whether the distributions of predictions between the two age groups are statistically different. A large KS statistic and small p-value would indicate significant distributional differences.

In [ ]:
# ks test between age groups
arr_pred_young = df_test.loc[df_test['str_age_group'] == f'<{int_age_threshold}', 'flt_pred'].values
arr_pred_old = df_test.loc[df_test['str_age_group'] == f'>={int_age_threshold}', 'flt_pred'].values

ks_result = ks_2samp(arr_pred_young, arr_pred_old)
print(f'KS Statistic: {ks_result.statistic:.4f}')
print(f'P-Value: {ks_result.pvalue:.4f}')